# MeaningFlow Notebook 01  
## Semantic Space → Coverage → Opportunity

#This notebook demonstrates the core MeaningFlow workflow:
1. Load semantic objects, demand signals, and structural edges
2. Construct a semantic graph
3. Measure demand-weighted semantic coverage
4. Identify high-value opportunity regions in meaning space

The goal is not prediction, but decision support.


In [1]:
# CELL 2 — Imports (Code)

import pandas as pd
import numpy as np
import networkx as nx

from pathlib import Path


In [2]:
#CELL 3 — Load Example Data (Code)

DATA_DIR = Path("../data/examples")

objects = pd.read_csv(DATA_DIR / "objects.csv")
demand = pd.read_csv(DATA_DIR / "demand.csv")
edges = pd.read_csv(DATA_DIR / "edges.csv")

objects.head(), demand.head(), edges.head()


(        id      type                                               text  \
 0  doc_001  DOCUMENT         Guide: Best trail running shoes for winter   
 1  doc_002  DOCUMENT              Product Category: Trail Running Shoes   
 2  doc_003  DOCUMENT  How to choose running shoes: stability vs neutral   
 3  qry_001     QUERY                    best trail running shoes winter   
 4  qry_002     QUERY                        trail running shoes near me   
 
                                        metadata_json  
 0  {"site_section":"blog","url":"/blog/winter-tra...  
 1  {"site_section":"category","url":"/trail-runni...  
 2  {"site_section":"blog","url":"/blog/stability-...  
 3                  {"source":"search","market":"US"}  
 4                  {"source":"search","market":"US"}  ,
   object_id    signal_type   value  start_date    end_date  \
 0   qry_001  SEARCH_VOLUME  5400.0  2025-10-01  2025-10-31   
 1   qry_002  SEARCH_VOLUME  2200.0  2025-10-01  2025-10-31   
 2   qry_003  SE

In [3]:
# CELL 4 — Basic Object Separation (Code)
documents = objects[objects["type"] == "DOCUMENT"]
queries = objects[objects["type"] == "QUERY"]
entities = objects[objects["type"] == "ENTITY"]

len(documents), len(queries), len(entities)


(3, 3, 3)

In [4]:
#CELL 5 — Build Semantic Graph (Code)

G = nx.DiGraph()

# Add nodes
for _, row in objects.iterrows():
    G.add_node(row["id"], type=row["type"])

# Add edges
for _, row in edges.iterrows():
    G.add_edge(
        row["src_id"],
        row["dst_id"],
        type=row["type"],
        weight=row["weight"]
    )

G.number_of_nodes(), G.number_of_edges()


(11, 11)

In [5]:
#CELL 6 — Demand Aggregation (Code)
query_demand = (
    demand[demand["signal_type"] == "SEARCH_VOLUME"]
    .groupby("object_id")["value"]
    .sum()
)

query_demand


object_id
qry_001    5400.0
qry_002    2200.0
qry_003    1600.0
Name: value, dtype: float64

In [6]:
#CELL: Config (Code)
USE_SENTENCE_TRANSFORMERS = True
MODEL_NAME = "all-MiniLM-L6-v2"
TOPK = 3
SIM_THRESHOLD = 0.55


### Step 3.1.0 — Prepare Text for Embeddings

Create `text_for_embedding` as the canonical text field used for embeddings.
If `text` is missing, fall back to `id` so every object can be embedded.


In [7]:
import json

def parse_json(x):
    try:
        return json.loads(x) if isinstance(x, str) and x.strip() else {}
    except Exception:
        return {}

# If your CSV has metadata_json, parse it (optional but useful later)
if "metadata_json" in objects.columns:
    objects["metadata"] = objects["metadata_json"].apply(parse_json)
else:
    objects["metadata"] = [{} for _ in range(len(objects))]

# Ensure a text column exists
if "text" not in objects.columns:
    objects["text"] = ""

# Build text_for_embedding
objects["text_for_embedding"] = objects["text"].fillna("")
objects.loc[objects["text_for_embedding"].str.strip() == "", "text_for_embedding"] = objects["id"].astype(str)

objects[["id", "type", "text", "text_for_embedding"]].head()


,id,type,text,text_for_embedding
0,doc_001,DOCUMENT,Guide: Best trail running shoes for winter,Guide: Best trail running shoes for winter
1,doc_002,DOCUMENT,Product Category: Trail Running Shoes,Product Category: Trail Running Shoes
2,doc_003,DOCUMENT,How to choose running shoes: stability vs neutral,How to choose running shoes: stability vs neutral
3,qry_001,QUERY,best trail running shoes winter,best trail running shoes winter
4,qry_002,QUERY,trail running shoes near me,trail running shoes near me


In [8]:
%pip install -q sentence-transformers scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [9]:
%pip install -q ipywidgets


Note: you may need to restart the kernel to use updated packages.


In [10]:
# Step 3.1.4 — Build Embeddings (robust on Windows/VS Code)
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Reduce noisy warnings / renderer issues
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

texts = objects["text_for_embedding"].tolist()

if USE_SENTENCE_TRANSFORMERS:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME)
    # Disable progress bar to avoid ipywidgets renderer issues in VS Code
    X = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
else:
    rng = np.random.default_rng(42)
    dim = 384
    X = rng.normal(size=(len(objects), dim))
    X = X / np.linalg.norm(X, axis=1, keepdims=True)

print("Embeddings shape:", X.shape)

id_to_idx = {obj_id: i for i, obj_id in enumerate(objects["id"].tolist())}
doc_ids = objects.loc[objects["type"] == "DOCUMENT", "id"].tolist()
qry_ids = objects.loc[objects["type"] == "QUERY", "id"].tolist()

doc_idx = np.array([id_to_idx[i] for i in doc_ids])
qry_idx = np.array([id_to_idx[i] for i in qry_ids])

S = cosine_similarity(X[qry_idx], X[doc_idx])
print("Similarity matrix shape:", S.shape)


Embeddings shape: (11, 384)
Similarity matrix shape: (3, 3)


In [11]:
# CELL: Add Similarity Edges (Code)
for qi, qid in enumerate(qry_ids):
    sims = S[qi]
    topk_idx = np.argsort(-sims)[:TOPK]
    for j in topk_idx:
        score = float(sims[j])
        if score >= SIM_THRESHOLD:
            did = doc_ids[j]
            G.add_edge(qid, did, type="SIMILARITY", weight=score)

G.number_of_edges()

14

In [12]:
# CELL: Coverage + Opportunity (Code)
query_demand = (
    demand[demand["signal_type"] == "SEARCH_VOLUME"]
    .groupby("object_id")["value"]
    .sum()
)
query_demand = query_demand[query_demand.index.isin(qry_ids)]

covered_ids = []
uncovered_rows = []

for qi, qid in enumerate(qry_ids):
    if qid not in query_demand.index:
        continue
    best_sim = float(S[qi].max())
    if best_sim >= SIM_THRESHOLD:
        covered_ids.append(qid)
    else:
        d = float(query_demand.loc[qid])
        gap = max(0.0, SIM_THRESHOLD - best_sim)
        uncovered_rows.append({
            "query_id": qid,
            "best_sim": best_sim,
            "demand": d,
            "gap": gap,
            "opportunity": d * gap
        })

coverage_ratio = (query_demand.loc[covered_ids].sum() / query_demand.sum()) if len(query_demand) else np.nan
coverage_ratio

np.float64(1.0)

In [13]:
#CELL: Display Ranked Opportunities (Code)
opportunity_df = pd.DataFrame(uncovered_rows).sort_values("opportunity", ascending=False)

opportunity_df = opportunity_df.merge(
    objects[objects["type"] == "QUERY"][["id", "text_for_embedding"]],
    left_on="query_id", right_on="id", how="left"
)

opportunity_df[["query_id","text_for_embedding","demand","best_sim","gap","opportunity"]]


KeyError: 'opportunity'

In [ ]:
#CELL 7 — Coverage Proxy (Simple v0)
def covered_queries(graph, doc_ids):
    covered = set()
    for q in query_demand.index:
        for d in doc_ids:
            if graph.has_edge(q, d) or graph.has_edge(d, q):
                covered.add(q)
    return covered

doc_ids = set(documents["id"])
covered = covered_queries(G, doc_ids)

coverage_ratio = (
    query_demand.loc[list(covered)].sum() / query_demand.sum()
)

coverage_ratio


NameError: name 'documents' is not defined

In [ ]:
#CELL 8 — Opportunity Scoring (Code)
uncovered = set(query_demand.index) - covered

opportunity = (
    query_demand.loc[list(uncovered)]
    .sort_values(ascending=False)
)

opportunity


In [ ]:
### Interpretation

- Covered queries represent semantic regions already supported by content.
- Uncovered queries represent demand-weighted opportunity.
- This simple proxy demonstrates how structure + demand identifies investment priorities.

In future iterations:
- Replace edge-based coverage with embedding distance
- Introduce authority weighting
- Simulate counterfactual content additions


In [ ]:
MeaningFlow reframes SEO and content optimization as a
**semantic capital allocation problem**.

Rather than optimizing pages, we identify where meaning,
structure, and demand are misaligned—and quantify the
economic opportunity of correcting that misalignment.

### Dependencies
This notebook optionally uses `sentence-transformers` for embeddings.
If unavailable, set `USE_SENTENCE_TRANSFORMERS = False` to run with deterministic random embeddings.


## Step 3.2 — Graph Metrics & Authority Flow

In this step, we analyze **structural properties** of the MeaningFlow graph.

While Step 3.1 identified *semantic demand gaps*, Step 3.2 answers:
- Where does authority concentrate?
- Which nodes act as bottlenecks?
- Which semantic objects are isolated or weakly connected?

This reveals how **structure amplifies or suppresses visibility**.


### Step 3.2.1 — Construct Structural Subgraphs

We analyze two complementary graphs:

1. **Link Graph**  
   - Internal links only  
   - Models authority flow and navigability

2. **Semantic Graph**  
   - Semantic similarity + entity relationships  
   - Models meaning connectivity and competition


In [ ]:
# Internal link graph (authority flow)
link_edges = [
    (u, v, d)
    for u, v, d in G.edges(data=True)
    if d.get("type") == "INTERNAL_LINK"
]

LinkGraph = nx.DiGraph()
LinkGraph.add_edges_from([(u, v) for u, v, _ in link_edges])

# Semantic graph (meaning connectivity)
semantic_edges = [
    (u, v, d)
    for u, v, d in G.edges(data=True)
    if d.get("type") in {"SIMILARITY", "ENTITY_MENTION", "CO_OCCURRENCE"}
]

SemanticGraph = nx.DiGraph()
SemanticGraph.add_edges_from([(u, v) for u, v, _ in semantic_edges])

LinkGraph.number_of_nodes(), SemanticGraph.number_of_nodes()


### Step 3.2.2 — Degree-Based Structure

Degree metrics identify:
- **Hubs** (high out-degree)
- **Receivers** (high in-degree)
- Structural imbalance

This is a first-pass diagnostic.


In [ ]:
degree_df = pd.DataFrame({
    "node": list(G.nodes()),
    "in_degree": [G.in_degree(n) for n in G.nodes()],
    "out_degree": [G.out_degree(n) for n in G.nodes()],
})

degree_df.sort_values("in_degree", ascending=False).head(10)


### Step 3.2.3 — PageRank Authority (Internal Links)

PageRank approximates how authority flows through internal structure.
Pages with high PageRank act as **visibility amplifiers**.


In [ ]:
if LinkGraph.number_of_nodes() > 0:
    pagerank = nx.pagerank(LinkGraph, alpha=0.85)
    pr_df = (
        pd.DataFrame.from_dict(pagerank, orient="index", columns=["pagerank"])
        .sort_values("pagerank", ascending=False)
        .reset_index()
        .rename(columns={"index": "node"})
    )
    pr_df.head(10)
else:
    pr_df = pd.DataFrame(columns=["node", "pagerank"])
    pr_df


### Step 3.2.4 — Bottleneck Detection (Betweenness Centrality)

Nodes with high betweenness act as **structural choke points**.
If they fail, large portions of semantic authority are cut off.


In [ ]:
if SemanticGraph.number_of_nodes() > 0:
    betweenness = nx.betweenness_centrality(SemanticGraph, normalized=True)
    btw_df = (
        pd.DataFrame.from_dict(betweenness, orient="index", columns=["betweenness"])
        .sort_values("betweenness", ascending=False)
        .reset_index()
        .rename(columns={"index": "node"})
    )
    btw_df.head(10)
else:
    btw_df = pd.DataFrame(columns=["node", "betweenness"])
    btw_df


### Step 3.2.5 — Isolates & Structural Gaps

Isolated or weakly connected nodes represent:
- Orphaned content
- Semantic dead-ends
- Unleveraged entities or topics


In [ ]:
isolates = list(nx.isolates(SemanticGraph))
isolates_df = pd.DataFrame({"isolated_node": isolates})
isolates_df


In [ ]:
object_lookup = objects.set_index("id")[["type", "text_for_embedding"]]

top_authority = pr_df.merge(object_lookup, left_on="node", right_index=True, how="left")
top_bottlenecks = btw_df.merge(object_lookup, left_on="node", right_index=True, how="left")
isolates_enriched = isolates_df.merge(object_lookup, left_on="isolated_node", right_index=True, how="left")

top_authority.head(10), top_bottlenecks.head(10), isolates_enriched.head(10)


### Interpretation

- **High PageRank nodes** act as authority hubs and should anchor key semantic regions.
- **High betweenness nodes** are structural bottlenecks; changes here have outsized impact.
- **Isolated nodes** indicate wasted semantic investment or missing internal structure.

Combined with Step 3.1:
- Demand gaps tell us *what to build*
- Structural metrics tell us *where to connect and reinforce*


MeaningFlow now models both:
- **Semantic geometry** (what content means)
- **Structural mediation** (how meaning flows)

This allows decisions that optimize:
- Coverage
- Authority
- Resilience

—not just rankings or traffic.


MeaningFlow now models both:
- **Semantic geometry** (what content means)
- **Structural mediation** (how meaning flows)

This allows decisions that optimize:
- Coverage
- Authority
- Resilience

—not just rankings or traffic.


In [ ]:
from pathlib import Path

EXPORT_DIR = Path("../outputs/exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_DIR


### Step 3.3.1 — Document Metadata

We extract interpretable segments from DOCUMENT metadata
(e.g., site_section) for coverage and opportunity summaries.


In [ ]:
import pandas as pd

docs_df = objects[objects["type"] == "DOCUMENT"].copy()

# If you have metadata already parsed into objects["metadata"], use it; otherwise parse metadata_json
if "metadata" in objects.columns:
    meta_series = docs_df["metadata"]
else:
    import json
    def parse_json(x):
        try:
            return json.loads(x) if isinstance(x, str) and x.strip() else {}
        except Exception:
            return {}
    meta_series = docs_df["metadata_json"].apply(parse_json)

docs_df["site_section"] = meta_series.apply(lambda d: d.get("site_section", "unknown"))

docs_df[["id", "site_section", "text_for_embedding"]].head()


### Step 3.3.2 — Query → Best Document Assignment

For each query, we identify the best-matching document by semantic similarity.
This allows us to attribute "coverage" and "misses" to site sections.


In [ ]:
# Build a lookup from doc index -> doc id
doc_ids = objects.loc[objects["type"] == "DOCUMENT", "id"].tolist()
qry_ids = objects.loc[objects["type"] == "QUERY", "id"].tolist()

# S is the query x doc similarity matrix from Step 3.1
best_doc_idx = S.argmax(axis=1)
best_doc_sim = S.max(axis=1)

query_best = pd.DataFrame({
    "query_id": qry_ids,
    "best_doc_id": [doc_ids[i] for i in best_doc_idx],
    "best_sim": best_doc_sim.astype(float)
})

# attach query demand
query_best = query_best.merge(
    query_demand.rename("demand"),
    left_on="query_id", right_index=True, how="left"
).fillna({"demand": 0.0})

# attach query text
query_best = query_best.merge(
    objects[objects["type"] == "QUERY"][["id","text_for_embedding"]],
    left_on="query_id", right_on="id", how="left"
).drop(columns=["id"])

query_best.head()


### Step 3.3.3 — Coverage by Site Section

We group queries by the site section of their best-matching document.
Coverage is demand-weighted: covered if best_sim ≥ SIM_THRESHOLD.


In [ ]:
doc_section = docs_df.set_index("id")["site_section"]

query_best["best_doc_section"] = query_best["best_doc_id"].map(doc_section).fillna("unknown")
query_best["covered"] = query_best["best_sim"] >= SIM_THRESHOLD
query_best["gap"] = np.maximum(0.0, SIM_THRESHOLD - query_best["best_sim"])
query_best["opportunity"] = query_best["demand"] * query_best["gap"]

coverage_by_section = (
    query_best.groupby("best_doc_section")
    .agg(
        total_demand=("demand","sum"),
        covered_demand=("demand", lambda x: x[query_best.loc[x.index, "covered"]].sum()),
        avg_best_sim=("best_sim","mean"),
        total_opportunity=("opportunity","sum"),
        n_queries=("query_id","count")
    )
    .reset_index()
)

coverage_by_section["coverage_ratio"] = np.where(
    coverage_by_section["total_demand"] > 0,
    coverage_by_section["covered_demand"] / coverage_by_section["total_demand"],
    np.nan
)

coverage_by_section.sort_values("total_opportunity", ascending=False)


### Step 3.3.4 — Opportunity by Entity Mentions

We use DOCUMENT → ENTITY edges to attribute opportunity to entity areas.
This answers: "Which entity domains are under-supported relative to demand?"


In [ ]:
# Extract doc->entity mention edges
doc_entity_edges = [
    (u, v)
    for u, v, d in G.edges(data=True)
    if d.get("type") == "ENTITY_MENTION"
]

doc_to_entities = {}
for doc_id, ent_id in doc_entity_edges:
    doc_to_entities.setdefault(doc_id, set()).add(ent_id)

# explode to a long-form mapping
rows = []
for doc_id, ents in doc_to_entities.items():
    for ent_id in ents:
        rows.append({"doc_id": doc_id, "entity_id": ent_id})

doc_entity_map = pd.DataFrame(rows)
doc_entity_map.head()


In [ ]:
# Attribute each query's opportunity to entities mentioned by its best doc
qb = query_best.copy()

qb = qb.merge(
    doc_entity_map,
    left_on="best_doc_id", right_on="doc_id", how="left"
)

# attach entity text
entity_lookup = objects[objects["type"] == "ENTITY"][["id","text_for_embedding"]].rename(
    columns={"id":"entity_id","text_for_embedding":"entity_text"}
)

qb = qb.merge(entity_lookup, on="entity_id", how="left")

opportunity_by_entity = (
    qb.groupby(["entity_id","entity_text"], dropna=False)
    .agg(
        total_demand=("demand","sum"),
        total_opportunity=("opportunity","sum"),
        avg_best_sim=("best_sim","mean"),
        n_queries=("query_id","count")
    )
    .reset_index()
    .sort_values("total_opportunity", ascending=False)
)

opportunity_by_entity.head(15)


## Step 3.4 — Exports & Executive Summary

In this step, we export MeaningFlow outputs to `outputs/exports/`:

- `opportunities.csv` — query-level opportunity ranking
- `coverage_by_section.csv` — demand-weighted coverage by site section
- `opportunity_by_entity.csv` — opportunity attribution by entity domain
- `graph_metrics.csv` — authority/bottleneck metrics (from Step 3.2)
- `executive_summary.md` — short decision memo

These exports make the repo immediately legible without running the notebook.


In [ ]:
# 1) Query-level opportunities (from Step 3.1)
# If opportunity_df exists from Step 3.1, use it; else build from query_best
if "opportunity_df" in globals():
    opportunities_export = opportunity_df.copy()
else:
    opportunities_export = query_best.loc[query_best["covered"] == False, [
        "query_id","text_for_embedding","demand","best_sim","gap","opportunity","best_doc_id","best_doc_section"
    ]].sort_values("opportunity", ascending=False)

opportunities_path = EXPORT_DIR / "opportunities.csv"
opportunities_export.to_csv(opportunities_path, index=False)

# 2) Coverage by site section
coverage_section_path = EXPORT_DIR / "coverage_by_section.csv"
coverage_by_section.to_csv(coverage_section_path, index=False)

# 3) Opportunity by entity
opportunity_entity_path = EXPORT_DIR / "opportunity_by_entity.csv"
opportunity_by_entity.to_csv(opportunity_entity_path, index=False)

opportunities_path, coverage_section_path, opportunity_entity_path


### Step 3.4.1 — Export Graph Metrics

We export the structural rankings from Step 3.2:
- PageRank authority (internal links)
- Betweenness bottlenecks (semantic graph)
- Isolates (semantic graph)


In [ ]:
# Build a unified graph metrics export table
rows = []

# PageRank authority
if "pr_df" in globals() and len(pr_df) > 0:
    for _, r in pr_df.head(100).iterrows():
        rows.append({"metric":"pagerank", "node": r["node"], "value": float(r["pagerank"])})

# Betweenness bottlenecks
if "btw_df" in globals() and len(btw_df) > 0:
    for _, r in btw_df.head(100).iterrows():
        rows.append({"metric":"betweenness", "node": r["node"], "value": float(r["betweenness"])})

# Isolates
if "isolates" in globals() and len(isolates) > 0:
    for n in isolates:
        rows.append({"metric":"isolate", "node": n, "value": 1.0})

graph_metrics_export = pd.DataFrame(rows)

# Enrich with type/text if possible
object_lookup = objects.set_index("id")[["type", "text_for_embedding"]]
graph_metrics_export = graph_metrics_export.merge(
    object_lookup, left_on="node", right_index=True, how="left"
)

graph_metrics_path = EXPORT_DIR / "graph_metrics.csv"
graph_metrics_export.to_csv(graph_metrics_path, index=False)

graph_metrics_path


### Step 3.4.2 — Executive Summary

We generate a short decision memo summarizing:
- Overall demand-weighted coverage
- Highest opportunity queries
- Most under-covered site sections
- Key authority hubs and bottlenecks

This memo is written to `outputs/exports/executive_summary.md`.


In [ ]:
top_queries = opportunities_export.head(5)[["query_id","text_for_embedding","demand","best_sim","opportunity"]]
top_sections = coverage_by_section.sort_values("total_opportunity", ascending=False).head(5)[
    ["best_doc_section","total_demand","coverage_ratio","total_opportunity","n_queries"]
]
top_authority_nodes = pr_df.head(5) if "pr_df" in globals() else pd.DataFrame()
top_bottlenecks_nodes = btw_df.head(5) if "btw_df" in globals() else pd.DataFrame()

memo_lines = []
memo_lines.append("# MeaningFlow Executive Summary\n")
memo_lines.append(f"**Demand-weighted semantic coverage:** `{coverage_ratio:.2%}`  \n")
memo_lines.append(f"**Similarity threshold:** `{SIM_THRESHOLD}` | **TopK:** `{TOPK}`  \n")

memo_lines.append("\n## Highest Opportunity Queries\n")
memo_lines.append(top_queries.to_markdown(index=False))

memo_lines.append("\n## Highest Opportunity Site Sections\n")
memo_lines.append(top_sections.to_markdown(index=False))

if len(top_authority_nodes) > 0:
    memo_lines.append("\n## Authority Hubs (Internal Links / PageRank)\n")
    memo_lines.append(top_authority_nodes.to_markdown(index=False))

if len(top_bottlenecks_nodes) > 0:
    memo_lines.append("\n## Bottlenecks (Semantic Graph / Betweenness)\n")
    memo_lines.append(top_bottlenecks_nodes.to_markdown(index=False))

memo_lines.append("\n## Notes\n")
memo_lines.append("- Coverage reflects semantic proximity between queries and documents, weighted by demand.\n")
memo_lines.append("- Opportunity highlights high-demand queries with weak semantic support.\n")
memo_lines.append("- Site sections and entity domains help prioritize where to build, improve, or consolidate.\n")

executive_summary_path = EXPORT_DIR / "executive_summary.md"
executive_summary_path.write_text("\n".join(memo_lines), encoding="utf-8")

executive_summary_path
